# 04 — Evaluate GPT-5 (Direct and Stepwise)

Evaluates GPT-5 on PlanBench plan verification using 2-shot prompting
in both direct and stepwise formats.

## Conditions

- **GPT-5 Direct** (`G5_direct`): full plan presented in a single prompt;
  model outputs a single valid/invalid verdict
- **GPT-5 Stepwise** (`G5_stepwise`): each action verified individually via
  multi-turn conversation; plan verdict derived from the pipeline

## Few-shot examples

2-shot prompting: 1 valid + 1 invalid plan per domain, loaded from
`paper2/data/planbench_direct_fewshot.jsonl` (built by `00_data_preparation.ipynb`).
Zero overlap with test set.

## Output

- `paper2/results/results_gpt5.json` — metrics for all conditions
- `paper2/results/raw_preds_gpt5.json` — raw prediction arrays

## Prerequisites

Run `00_data_preparation.ipynb` first. No GPU required.

## 1. Setup

Loads OpenAI API key from Colab secrets (`OPENAI_API_KEY`).
DEBUG mode evaluates only 3 plans per domain.

In [ ]:
DEBUG   = False
DEBUG_N = 3
print(f'Mode: {"DEBUG" if DEBUG else "FULL EVAL"}')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip -q install -U scikit-learn openai


In [ ]:
try:
    from google.colab import userdata
    from huggingface_hub import login
    login(token=userdata.get('HF_TOKEN'))
    print('Logged in via HF_TOKEN secret')
except Exception:
    from huggingface_hub import login
    login()

In [ ]:
import os, json, re, random, warnings, time
import numpy as np
import copy
from collections import Counter, defaultdict
import torch

from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
import openai

warnings.filterwarnings('ignore')
SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

ROOT        = '/content/drive/MyDrive/paper_codes/paper2'
DATA_DIR    = f'{ROOT}/data'
RESULTS_DIR = f'{ROOT}/results'
os.makedirs(RESULTS_DIR, exist_ok=True)

SW_TEST_PATH = os.path.join(DATA_DIR, 'planbench_stepwise_test.jsonl')
DI_TEST_PATH     = os.path.join(DATA_DIR, 'planbench_direct_test.jsonl')
DI_FEWSHOT_PATH  = os.path.join(DATA_DIR, 'planbench_direct_fewshot.jsonl')

MODEL_NAME     = 'google/flan-t5-xl'
GPT5_MODEL     = 'gpt-5.4'  # GPT-5.4: OpenAI flagship as of March 2026
MAX_SOURCE_LEN = 1024
MAX_TARGET_LEN = 4
BATCH_SIZE     = 8

DOMAINS = ['blocksworld', 'blocksworld_3', 'mystery', 'mystery_3', 'logistics']
MERGE_GROUPS = {
    'blocksworld': ['blocksworld', 'blocksworld_3'],
    'mystery'    : ['mystery', 'mystery_3'],
    'logistics'  : ['logistics'],
}


# OpenAI API key
try:
    from google.colab import userdata
    openai.api_key = userdata.get('OPENAI_API_KEY')
    print('OpenAI API key loaded from Colab secrets')
except Exception:
    openai.api_key = os.environ.get('OPENAI_API_KEY', '')
    print('OpenAI API key loaded from environment')


## 2. Prompt Builders

Same stepwise and direct prompt formats as all other notebooks.

In [ ]:
# Identical to Phase 1 build_stepwise_prompt
def build_stepwise_prompt(rules, state_facts, conclusion):
    premises = rules + state_facts
    prem_block = '\n'.join('- ' + p.strip().rstrip('.') + '.' for p in premises if p.strip())
    return (
        'Task: Determine whether the conclusion is entailed or contradicted given the premises.\n'
        'Premises:\n' + prem_block + '\n\n'
        'Conclusion:\n' + conclusion.strip() + '\n\n'
        'Output format: Answer A if the conclusion is entailed, B if it is contradicted.\n'
        'Answer:'
    )

# Identical to Phase 1 build_direct_prompt
def build_direct_prompt(rules, init_facts, goal_facts, actions):
    rules_block = '\n'.join('- ' + r.strip().rstrip('.') + '.' for r in rules if r.strip())
    state_block = '\n'.join('- ' + f.strip().rstrip('.') + '.' for f in init_facts if f.strip())
    goal_block  = '\n'.join('- ' + g.strip().rstrip('.') + '.' for g in goal_facts if g.strip())
    plan_block  = '\n'.join(f'{i+1}. {a.strip()}' for i, a in enumerate(actions))
    return (
        'Task: Determine whether the following plan is valid given the rules and initial state.\n\n'
        'Rules:\n' + rules_block + '\n\n'
        'Initial state:\n' + state_block + '\n\n'
        'Goal:\n' + goal_block + '\n\n'
        'Plan:\n' + plan_block + '\n\n'
        'Output format: Answer A if the plan is valid, B if it is not.\n'
        'Answer:'
    )

# Extended direct prompt for GPT-4: same body, extended output format
def build_direct_prompt_gpt4(rules, init_facts, goal_facts, actions):
    rules_block = '\n'.join('- ' + r.strip().rstrip('.') + '.' for r in rules if r.strip())
    state_block = '\n'.join('- ' + f.strip().rstrip('.') + '.' for f in init_facts if f.strip())
    goal_block  = '\n'.join('- ' + g.strip().rstrip('.') + '.' for g in goal_facts if g.strip())
    plan_block  = '\n'.join(f'{i+1}. {a.strip()}' for i, a in enumerate(actions))
    return (
        'Task: Determine whether the following plan is valid given the rules and initial state.\n\n'
        'Rules:\n' + rules_block + '\n\n'
        'Initial state:\n' + state_block + '\n\n'
        'Goal:\n' + goal_block + '\n\n'
        'Plan:\n' + plan_block + '\n\n'
        'Output format: If the plan is valid, answer exactly: Answer: A, Step: 0\n'
        'If the plan is invalid, answer exactly: Answer: B, Step: N '
        'where N is the step number of the first invalid action.\n'
        'Answer:'
    )

def normalize_pred(text):
    if not text: return 'B'
    c = text.strip()[0].upper()
    return c if c in {'A', 'B'} else 'B'

print('Prompt builders: OK')

## 3. State Helpers and Goal Check

Identical to `02_eval_flant5.ipynb` and `03_eval_gpt41mini.ipynb`.

In [ ]:
# All state helpers copied verbatim from Phase 3

def _bw_parse(facts):
    s = {'on': [], 'on_table': [], 'clear': [], 'holding': None, 'hand_empty': True}
    for f in facts:
        fl = f.lower().rstrip('.')
        m = re.search(r'the (\w+) block is on top of the (\w+) block', fl)
        if m: s['on'].append((m.group(1), m.group(2))); continue
        m = re.search(r'the (\w+) block is on the table', fl)
        if m: s['on_table'].append(m.group(1)); continue
        m = re.search(r'the (\w+) block is clear', fl)
        if m: s['clear'].append(m.group(1)); continue
        m = re.search(r'holding (?:the )?(\w+) block', fl)
        if m: s['holding'] = m.group(1); continue
        if 'hand is empty' in fl: s['hand_empty'] = True
    return s

def _bw_apply(state, action):
    s = copy.deepcopy(state); a = action.strip().lower()
    m = re.match(r'unstack (?:the )?(\w+) block from (?:on top of )?(?:the )?(\w+) block', a)
    if m:
        X, Y = m.group(1), m.group(2)
        s['on'] = [(a2, b2) for a2, b2 in s['on'] if not (a2 == X and b2 == Y)]
        s['clear'] = [c for c in s['clear'] if c != X]
        s['hand_empty'] = False; s['holding'] = X
        if Y not in s['clear']: s['clear'].append(Y)
        return s
    m = re.match(r'pick up (?:the )?(\w+) block', a)
    if m:
        X = m.group(1); s['on_table'] = [b for b in s['on_table'] if b != X]
        s['clear'] = [c for c in s['clear'] if c != X]
        s['hand_empty'] = False; s['holding'] = X; return s
    m = re.match(r'put down (?:the )?(\w+) block', a)
    if m:
        X = m.group(1); s['holding'] = None; s['hand_empty'] = True
        if X not in s['on_table']: s['on_table'].append(X)
        if X not in s['clear']: s['clear'].append(X); return s
    m = re.match(r'stack (?:the )?(\w+) block on (?:top of )?(?:the )?(\w+) block', a)
    if m:
        X, Y = m.group(1), m.group(2); s['holding'] = None; s['hand_empty'] = True
        s['clear'] = [c for c in s['clear'] if c != Y]
        if X not in s['clear']: s['clear'].append(X)
        if (X, Y) not in s['on']: s['on'].append((X, Y)); return s
    return None

MYSTERY_FULL_MAP = {
    'attack': 'action1', 'succumb': 'action2', 'overcome': 'action3', 'feast': 'action4',
    'Attack': 'Action1', 'Succumb': 'Action2', 'Overcome': 'Action3', 'Feast': 'Action4',
    'Province': 'Rel1', 'province': 'rel1', 'Planet': 'Rel2', 'planet': 'rel2',
    'Harmony': 'Rel3', 'harmony': 'rel3', 'Pain': 'Rel4', 'pain': 'rel4',
    'Craves': 'Rel5', 'craves': 'rel5',
}

def abstract_mystery(text):
    result = text
    for original, replacement in MYSTERY_FULL_MAP.items():
        result = re.sub(r'\b' + re.escape(original) + r'\b', replacement, result)
    return result

def _mystery_apply(state, action):
    a = action.strip().lower(); s = set(state)
    m = re.match(r'action1 object (\w+)$', a)
    if m:
        x = m.group(1); s -= {f'rel1 object {x}', f'rel2 object {x}', 'rel3'}
        s.add(f'rel4 object {x}'); return s
    m = re.match(r'action2 object (\w+)$', a)
    if m:
        x = m.group(1); s.discard(f'rel4 object {x}')
        s |= {f'rel1 object {x}', f'rel2 object {x}', 'rel3'}; return s
    m = re.match(r'action3 object (\w+) from object (\w+)$', a)
    if m:
        x, y = m.group(1), m.group(2)
        s -= {f'rel1 object {y}', f'rel4 object {x}'}
        s |= {'rel3', f'rel1 object {x}', f'object {x} rel5 object {y}'}; return s
    m = re.match(r'action4 object (\w+) from object (\w+)$', a)
    if m:
        x, y = m.group(1), m.group(2)
        s -= {f'object {x} rel5 object {y}', f'rel1 object {x}', 'rel3'}
        s |= {f'rel4 object {x}', f'rel1 object {y}'}; return s
    return None

def _logistics_parse(facts):
    locs = {}; airports = set(); city_map = {}
    for f in facts:
        fl = f.lower().rstrip('.')
        if 'is an airport' in fl:
            m = re.search(r'(\S+) is an airport', fl)
            if m: airports.add(m.group(1))
        elif 'is in city' in fl:
            m = re.search(r'(\S+) is in city (\S+)', fl)
            if m: city_map[m.group(1)] = m.group(2)
        elif 'is at' in fl:
            m = re.search(r'(\S+) is at (\S+)', fl)
            if m: locs[m.group(1)] = m.group(2)
        elif 'is in' in fl:
            m = re.search(r'(\S+) is in (\S+)', fl)
            if m: locs[m.group(1)] = m.group(2)
    return {'locs': locs, 'airports': airports, 'city_map': city_map}

def _logistics_apply(state, action):
    s = copy.deepcopy(state); a = action.strip().lower()
    m = re.match(r'load (\S+) into (\S+) at (\S+)', a)
    if m: s['locs'][m.group(1)] = m.group(2); return s
    m = re.match(r'unload (\S+) from (\S+) at (\S+)', a)
    if m: s['locs'][m.group(1)] = m.group(3); return s
    m = re.match(r'drive (\S+) from (\S+) to (\S+) in (\S+)', a)
    if m: s['locs'][m.group(1)] = m.group(3); return s
    m = re.match(r'fly (\S+) from (\S+) to (\S+)', a)
    if m: s['locs'][m.group(1)] = m.group(3); return s
    return None

def _to_fact_set(state, domain):
    if domain in ('blocksworld', 'blocksworld_3'):
        facts = []
        for x, y in state['on']:     facts.append(f'the {x} block is on top of the {y} block')
        for x in state['on_table']:  facts.append(f'the {x} block is on the table')
        for x in state['clear']:     facts.append(f'the {x} block is clear')
        if state['holding']:         facts.append(f'you are holding the {state["holding"]} block')
        if state['hand_empty']:      facts.append('the hand is empty')
        return frozenset(facts)
    elif domain in ('mystery', 'mystery_3'):
        return frozenset(state)
    elif domain == 'logistics':
        facts = []
        for e, loc in state['locs'].items():
            facts.append(f'{e} is in {loc}'
                         if any(loc.startswith(v) for v in ('truck', 'airplane'))
                         else f'{e} is at {loc}')
        return frozenset(facts)
    return frozenset()

def goal_check_from_records(plan_recs, domain, goal_facts):
    if domain in ('blocksworld', 'blocksworld_3'):
        state = _bw_parse(plan_recs[0]['state_facts'])
        apply_fn = _bw_apply
    elif domain in ('mystery', 'mystery_3'):
        state = {f.lower().rstrip('.') for f in plan_recs[0]['state_facts']}
        apply_fn = _mystery_apply
    elif domain == 'logistics':
        state = _logistics_parse(plan_recs[0]['state_facts'])
        apply_fn = _logistics_apply
    else: return 0
    for rec in sorted(plan_recs, key=lambda x: x['step_idx']):
        action = rec['action']
        if domain in ('mystery', 'mystery_3'):
            action = abstract_mystery(action)
        ns = apply_fn(state, action)
        if ns is not None: state = ns
    fs = _to_fact_set(state, domain)
    return 1 if all(g.lower().rstrip('.') in fs for g in goal_facts) else 0

print('State helpers: OK')

## 4. Load Test Datasets

Same test files as `02_eval_flant5.ipynb` from `paper2/data/`.

In [ ]:
sw_all = [json.loads(l) for l in open(SW_TEST_PATH)]
di_all = [json.loads(l) for l in open(DI_TEST_PATH)]
print(f'Stepwise test records : {len(sw_all)}')
print(f'Direct   test records : {len(di_all)}')

sw_by_domain = defaultdict(list)
di_by_domain = defaultdict(list)
for r in sw_all: sw_by_domain[r['domain']].append(r)
for r in di_all: di_by_domain[r['domain']].append(r)

for d in DOMAINS:
    sw_cnt = Counter(r['label'] for r in sw_by_domain[d])
    di_cnt = Counter(r['label'] for r in di_by_domain[d])
    n_plans = len({r['plan_id'] for r in sw_by_domain[d]})
    print(f'{d:15s}  plans={n_plans:4d} | '
          f'sw={len(sw_by_domain[d])} (A={sw_cnt["A"]} B={sw_cnt["B"]}) | '
          f'di={len(di_by_domain[d])} (A={di_cnt["A"]} B={di_cnt["B"]})')

if DEBUG:
    for d in DOMAINS:
        # Guarantee at least 1 valid + 1 invalid plan in sw slice too
        _sw_valid   = [r['plan_id'] for r in sw_by_domain[d] if r['plan_label'] == 1]
        _sw_invalid = [r['plan_id'] for r in sw_by_domain[d] if r['plan_label'] == 0]
        _sw_keep = set()
        if _sw_valid:   _sw_keep.add(_sw_valid[0])
        if _sw_invalid: _sw_keep.add(_sw_invalid[0])
        _sw_extra = [pid for pid in list({r['plan_id'] for r in sw_by_domain[d]})
                     if pid not in _sw_keep][:max(0, DEBUG_N - 2)]
        _sw_keep.update(_sw_extra)
        sw_by_domain[d] = [r for r in sw_by_domain[d] if r['plan_id'] in _sw_keep]
        # Guarantee at least 1 valid (label_int=1) and 1 invalid (label_int=0)
        # plan in di_by_domain so few-shot selection never fails
        _valid_ids   = [r['plan_id'] for r in di_by_domain[d] if r['label_int'] == 1]
        _invalid_ids = [r['plan_id'] for r in di_by_domain[d] if r['label_int'] == 0]
        assert _valid_ids,   f'No valid di plans for {d}'
        assert _invalid_ids, f'No invalid di plans for {d}'
        _keep_ids = {_valid_ids[0], _invalid_ids[0]}
        # Add more plans up to DEBUG_N total
        _extra = [pid for pid in list({r['plan_id'] for r in di_by_domain[d]})
                  if pid not in _keep_ids][:max(0, DEBUG_N - 2)]
        _keep_ids.update(_extra)
        di_by_domain[d] = [r for r in di_by_domain[d] if r['plan_id'] in _keep_ids]
    print(f'[DEBUG] sliced to {DEBUG_N} plans per domain')

## 5. Few-Shot Example Selection

Loaded from `planbench_direct_fewshot.jsonl` — same examples as `03_eval_gpt41mini.ipynb`,
ensuring consistent few-shot prompting across all GPT evaluations.

In [ ]:
# Select 1 valid + 1 invalid plan per domain from the test set for 2-shot prompting.
# These plan_ids are excluded from G_direct evaluation to prevent leakage.
# For G_stepwise, the same plan_ids are used — valid plans have all-A stepwise
# records in the test set, and the invalid plan is guaranteed action_fail so it
# has a B-label step to demonstrate precondition violation.

gpt4_direct_examples   = {}   # domain -> {'valid': di_record, 'invalid': di_record}
gpt4_stepwise_examples = {}   # domain -> {'valid': [sw_records], 'invalid': [sw_records]}
gpt4_direct_exclude    = {}   # domain -> set of plan_ids excluded from G_direct test

for d in DOMAINS:
    di_recs = di_by_domain[d]
    sw_recs = sw_by_domain[d]

    # Direct examples: 1 valid + 1 invalid (action_fail preferred)
    valid_di   = next((r for r in di_recs if r['label_int'] == 1), None)
    invalid_di = next((r for r in di_recs
                       if r['label_int'] == 0 and r['fail_reason'] == 'action_fail'), None)
    if invalid_di is None:
        invalid_di = next((r for r in di_recs if r['label_int'] == 0), None)
    assert valid_di   is not None, f'No valid direct plan found for domain {d}'
    assert invalid_di is not None, f'No invalid direct plan found for domain {d}'

    gpt4_direct_examples[d] = {'valid': valid_di, 'invalid': invalid_di}
    gpt4_direct_exclude[d]  = {valid_di['plan_id'], invalid_di['plan_id']}

    # Stepwise examples: same plan_ids, looked up in sw_by_domain (test set)
    valid_pid   = valid_di['plan_id']
    invalid_pid = invalid_di['plan_id']
    valid_sw   = sorted([r for r in sw_recs if r['plan_id'] == valid_pid],
                        key=lambda x: x['step_idx'])
    invalid_sw = sorted([r for r in sw_recs if r['plan_id'] == invalid_pid],
                        key=lambda x: x['step_idx'])
    # Safety: if invalid sw steps have no B label, fall back to any B-label plan
    if not any(s['label'] == 'B' for s in invalid_sw):
        b_plans = {r['plan_id'] for r in sw_recs if r['label'] == 'B'}
        b_plans -= {valid_pid}
        if b_plans:
            fb_id      = next(iter(b_plans))
            invalid_sw = sorted([r for r in sw_recs if r['plan_id'] == fb_id],
                                 key=lambda x: x['step_idx'])

    gpt4_stepwise_examples[d] = {'valid': valid_sw, 'invalid': invalid_sw}
    print(f'{d:15s}: valid_pid={valid_pid} invalid_pid={invalid_pid} '
          f'sw_valid_steps={len(valid_sw)} sw_invalid_steps={len(invalid_sw)}')

print('\nFew-shot examples selected from test set (excluded from G_direct evaluation)')

## 6. GPT-5.4 API Helper

In [ ]:
# Single shared client — created once, reused for all calls
_openai_client = None
def get_client():
    global _openai_client
    if _openai_client is None:
        _openai_client = openai.OpenAI(api_key=openai.api_key)
    return _openai_client

def call_gpt4(messages, max_retries=10, base_delay=60):
    """Call GPT-5.4 with exponential backoff for rate limits.
    base_delay=60s because OpenAI rate limit windows are per-minute.
    Retries up to max_retries=10 times with exponential backoff.
    Returns empty string only after all retries exhausted.
    """
    client = get_client()
    for attempt in range(max_retries):
        try:
            response = client.chat.completions.create(
                model=GPT5_MODEL,
                messages=messages,
                max_completion_tokens=20,  # gpt-5.4 requires max_completion_tokens
            )
            return response.choices[0].message.content.strip()
        except openai.RateLimitError as e:
            # insufficient_quota = billing issue, retrying is pointless
            body = e.body if hasattr(e, 'body') and e.body else {}
            if isinstance(body, dict) and body.get('code') == 'insufficient_quota':
                raise RuntimeError(
                    'OpenAI account has no credits. '
                    'Add credits at platform.openai.com/settings/organization/billing'
                ) from e
            # Parse retry-after header if available
            wait = base_delay * (2 ** min(attempt, 3))  # cap at 8x base = 480s
            if hasattr(e, 'response') and e.response is not None:
                ra = e.response.headers.get('retry-after')
                if ra:
                    wait = max(int(ra) + 2, wait)
            print(f'  [Rate limit] waiting {wait}s (attempt {attempt+1}/{max_retries})')
            time.sleep(wait)
        except openai.APIStatusError as e:
            if e.status_code in (500, 502, 503, 529):  # transient server errors
                wait = base_delay * (2 ** min(attempt, 2))
                print(f'  [Server error {e.status_code}] waiting {wait}s (attempt {attempt+1}/{max_retries})')
                time.sleep(wait)
            else:
                print(f'  [API error {e.status_code}] {e.body} — not retrying')
                return ''  # non-retryable (auth, bad request, etc)
        except Exception as e:
            wait = base_delay
            print(f'  [API error] {e} (attempt {attempt+1}/{max_retries}), waiting {wait}s')
            time.sleep(wait)
    print(f'  [FAILED] all {max_retries} retries exhausted — recording empty response')
    return ''

def parse_gpt4_direct_response(text):
    """
    Parse structured GPT-4 direct response.
    Expected format: 'Answer: A, Step: 0' or 'Answer: B, Step: N'
    Returns (label: str 'A'/'B', step: int or None)
    """
    if not text:
        return 'EMPTY', None  # explicit marker — API call failed
    # Extract answer letter
    m_ans = re.search(r'Answer:\s*([AB])', text, re.IGNORECASE)
    label = m_ans.group(1).upper() if m_ans else 'B'
    # Extract step number
    m_step = re.search(r'Step:\s*(\d+)', text, re.IGNORECASE)
    step = int(m_step.group(1)) if m_step else None
    return label, step

def parse_gpt4_stepwise_response(text):
    """Parse GPT-4 stepwise response — same as normalize_pred."""
    return normalize_pred(text)

print('GPT-4 API helper: OK')

## 7. G5_direct Few-Shot Prompt Builder

In [ ]:
def build_gpt4_direct_messages(domain, test_record):
    """
    Build GPT-5.4 chat messages for G5_direct.
    System message sets the task.
    Two few-shot turns (valid + invalid) from gpt4_direct_examples[domain].
    Final user turn is the test record.
    The prompt body uses build_direct_prompt_gpt4 — identical structure to D1
    except the output format line asks for structured step output.
    """
    ex = gpt4_direct_examples[domain]

    def record_to_prompt(r):
        return build_direct_prompt_gpt4(
            r['rules'], r['init_facts'], r['goal_facts'], r['actions']
        )

    def record_to_answer(r):
        if r['label_int'] == 1:
            return 'Answer: A, Step: 0'
        else:
            # For the few-shot invalid example, the failing step comes from
            # the stepwise records: it is the step where label='B'
            sw_steps = gpt4_stepwise_examples[domain]['invalid']
            failing = next((s['step_num'] for s in sw_steps if s['label'] == 'B'), None)
            step_str = str(failing) if failing is not None else '1'
            return f'Answer: B, Step: {step_str}'

    messages = [
        {
            'role': 'system',
            'content': (
                'You are a precise logical reasoning assistant. '
                'You determine whether a plan is valid given rules and an initial state. '
                'Always respond in exactly the format specified.'
            )
        },
        # Few-shot example 1: valid plan
        {'role': 'user',      'content': record_to_prompt(ex['valid'])},
        {'role': 'assistant', 'content': record_to_answer(ex['valid'])},
        # Few-shot example 2: invalid plan
        {'role': 'user',      'content': record_to_prompt(ex['invalid'])},
        {'role': 'assistant', 'content': record_to_answer(ex['invalid'])},
        # Test query
        {'role': 'user',      'content': record_to_prompt(test_record)},
    ]
    return messages

print('G_direct few-shot prompt builder: OK')

## 8. G5_stepwise Few-Shot Prompt Builder

In [ ]:
def build_gpt4_stepwise_messages(domain, rules, state_facts, conclusion,
                                  conversation_history=None):
    """
    Build GPT-5.4 chat messages for G5_stepwise.
    First call: system + 2 few-shot turns (one valid step, one invalid step) + test step.
    Subsequent calls: conversation_history already contains prior turns, just append new step.
    The step prompt body uses build_stepwise_prompt — identical to S1/S2_fs.
    """
    step_prompt = build_stepwise_prompt(rules, state_facts, conclusion)

    if conversation_history is not None:
        # Continuation: append the new step to existing history
        return conversation_history + [{'role': 'user', 'content': step_prompt}]

    # First call: build full message list with few-shot examples
    ex = gpt4_stepwise_examples[domain]

    # Use first step of valid plan as positive example
    valid_step   = ex['valid'][0]   if ex['valid']   else None
    # Use the failing step of invalid plan as negative example
    invalid_step = next((s for s in ex['invalid'] if s['label'] == 'B'), None)
    if invalid_step is None and ex['invalid']:
        invalid_step = ex['invalid'][-1]

    messages = [
        {
            'role': 'system',
            'content': (
                'You are a precise logical reasoning assistant. '
                'You determine whether a conclusion is entailed or contradicted '
                'given premises. Always respond with a single letter: A or B.'
            )
        }
    ]

    if valid_step:
        messages += [
            {'role': 'user',      'content': valid_step['prompt']},
            {'role': 'assistant', 'content': 'A'},
        ]
    if invalid_step:
        messages += [
            {'role': 'user',      'content': invalid_step['prompt']},
            {'role': 'assistant', 'content': 'B'},
        ]

    messages.append({'role': 'user', 'content': step_prompt})
    return messages

print('G_stepwise few-shot prompt builder: OK')

In [ ]:
# ── G_direct evaluator ────────────────────────────────────────────────────────

def evaluate_g_direct(di_records, domain):
    """
    Evaluates G_direct on direct records, excluding few-shot example plans.
    Metrics:
      - plan_acc, plan_f1 (same as D1)
      - action_acc, action_f1: derived from GPT-4's predicted failing step
      - failing_step_acc: among invalid plans where gold failing step is known,
        what fraction does GPT-4 identify correctly
    """
    exclude = gpt4_direct_exclude[domain]
    records = [r for r in di_records if r['plan_id'] not in exclude]
    print(f'  [G5_direct|{domain}] n={len(records)} plans '
          f'(excluded {len([r for r in di_records if r["plan_id"] in exclude])} few-shot plans)')

    plan_golds, plan_preds_out = [], []
    # For action-level: treat each step in the plan as a prediction
    # Gold: all steps before failing_step are A(1), failing_step is B(0)
    # Pred: steps before predicted failing step are A(1), predicted failing step is B(0)
    action_golds_all, action_preds_all = [], []
    failing_step_golds, failing_step_preds = [], []

    # We need stepwise records to know n_actions per plan and gold failing steps
    sw_plans = defaultdict(list)
    for r in sw_by_domain[domain]:
        sw_plans[r['plan_id']].append(r)

    # Progress checkpoint path — saves after every plan so you can resume
    ckpt_path = os.path.join(RESULTS_DIR, f'ckpt_G5_direct_{domain}.json')
    done_pids = {}
    if os.path.exists(ckpt_path):
        done_pids = {r["plan_id"]: r for r in json.load(open(ckpt_path))}
        print(f'    Resuming from checkpoint: {len(done_pids)} plans already done')

    for rec in records:
        if rec["plan_id"] in done_pids:
            # Replay from checkpoint
            saved = done_pids[rec["plan_id"]]
            label, step = saved["label"], saved["step"]
        else:
            messages   = build_gpt4_direct_messages(domain, rec)
            response   = call_gpt4(messages)
            label, step = parse_gpt4_direct_response(response)
            # Save to checkpoint
            done_pids[rec["plan_id"]] = {"plan_id": rec["plan_id"],
                                          "label": label, "step": step}
            with open(ckpt_path, "w") as _f:
                json.dump(list(done_pids.values()), _f)

        if label == 'EMPTY':
            print(f'  [WARN] Empty API response for plan_id={rec["plan_id"]} '
                  f'domain={domain} — skipping this record')
            continue  # skip rather than corrupt results
        plan_gold = rec['label_int']
        plan_pred = 1 if label == 'A' else 0
        plan_golds.append(plan_gold)
        plan_preds_out.append(plan_pred)

        # Action-level reconstruction
        sw_steps = sorted(sw_plans.get(rec['plan_id'], []), key=lambda x: x['step_idx'])
        n_steps  = len(sw_steps)
        if n_steps > 0:
            # Gold action labels from stepwise records
            for s in sw_steps:
                action_golds_all.append(s['label_int'])
            # Predicted action labels: if GPT-4 says valid (A), all steps are A
            # If GPT-4 says invalid with step N, steps <N are A, step N is B
            if plan_pred == 1 or step is None or step == 0:
                action_preds_all.extend([1] * n_steps)
            else:
                for s in sw_steps:
                    if s['step_num'] < step:
                        action_preds_all.append(1)
                    elif s['step_num'] == step:
                        action_preds_all.append(0)
                    else:
                        # Steps after predicted fail: gold may or may not exist
                        # We don't evaluate these (break at first predicted fail)
                        # but we still recorded the gold above — align with dummy pred
                        action_preds_all.append(1)  # these cancel out in truncated eval

        # Failing step accuracy: only for truly invalid plans with a known gold step
        gold_fail = next((s['step_num'] for s in sw_steps if s['label_int'] == 0), None)
        if plan_gold == 0 and gold_fail is not None:
            failing_step_golds.append(gold_fail)
            failing_step_preds.append(step if step is not None else -1)

    acc_plan = accuracy_score(plan_golds, plan_preds_out)
    f1_plan  = f1_score(plan_golds, plan_preds_out, average='macro', zero_division=0)
    cm_plan  = confusion_matrix(plan_golds, plan_preds_out, labels=[0, 1])

    acc_act = accuracy_score(action_golds_all, action_preds_all) if action_golds_all else None
    f1_act  = f1_score(action_golds_all, action_preds_all, average='macro', zero_division=0) if action_golds_all else None
    cm_act  = confusion_matrix(action_golds_all, action_preds_all, labels=[0, 1]) if action_golds_all else None

    fs_acc = (sum(g == p for g, p in zip(failing_step_golds, failing_step_preds))
              / max(1, len(failing_step_golds))) if failing_step_golds else None

    print(f'    plan_acc={acc_plan:.4f} plan_f1={f1_plan:.4f} '
          f'failing_step_acc={fs_acc:.4f}' if fs_acc is not None else
          f'    plan_acc={acc_plan:.4f} plan_f1={f1_plan:.4f}')

    return {
        'cond': 'G5_direct', 'domain': domain, 'eval_type': 'direct',
        'n_plans': len(plan_golds),
        'plan_acc': round(acc_plan, 4), 'plan_f1': round(f1_plan, 4),
        'plan_cm': cm_plan.tolist(),
        'action_acc': round(acc_act, 4) if acc_act is not None else None,
        'action_f1':  round(f1_act,  4) if f1_act  is not None else None,
        'action_cm':  cm_act.tolist()    if cm_act  is not None else None,
        'failing_step_acc': round(fs_acc, 4) if fs_acc is not None else None,
        'n_failing_step_evaluated': len(failing_step_golds),
        'plan_golds': plan_golds, 'plan_preds': plan_preds_out,
        'action_golds': action_golds_all, 'action_preds': action_preds_all,
        'failing_step_golds': failing_step_golds,
        'failing_step_preds': failing_step_preds,
    }

print('G_direct evaluator: OK')

In [ ]:
# ── G_stepwise evaluator ──────────────────────────────────────────────────────

def evaluate_g_stepwise(sw_records, domain):
    """
    Evaluates G_stepwise.
    - Symbolic state update after each valid step (identical to S1/S2_fs).
    - GPT-4 only predicts valid/invalid per step.
    - Maintains conversation history per plan for multi-turn coherence.
    - No exclusion needed: stepwise examples are single steps, not full plans.
    """
    plans = defaultdict(list)
    for r in sw_records: plans[r['plan_id']].append(r)
    for pid in plans: plans[pid].sort(key=lambda x: x['step_idx'])
    plan_ids = sorted(plans.keys())

    print(f'  [G5_stepwise|{domain}] {len(plan_ids)} plans ...')

    plan_golds, plan_preds_out = [], []
    action_golds, action_preds = [], []
    err_action, err_goal = 0, 0

    # Progress checkpoint
    ckpt_path = os.path.join(RESULTS_DIR, f'ckpt_G5_stepwise_{domain}.json')
    done_pids_sw = {}
    if os.path.exists(ckpt_path):
        done_pids_sw = {r["plan_id"]: r for r in json.load(open(ckpt_path))}
        print(f'    Resuming from checkpoint: {len(done_pids_sw)} plans already done')

    for plan_idx, pid in enumerate(plan_ids):
        recs       = plans[pid]
        plan_gold  = recs[0]['plan_label']
        goal_facts = recs[0]['goal_facts']
        plan_pred  = 1; stopped = False; step_mistake = False
        history    = None  # conversation history for this plan

        for rec in recs:
            sgold = rec['label_int']
            # Build messages — first step builds full history with few-shot examples,
            # subsequent steps append to existing history
            messages = build_gpt4_stepwise_messages(
                domain,
                rec['rules'],
                rec['state_facts'],
                rec['conclusion'],
                conversation_history=history,
            )
            response = call_gpt4(messages)
            pred     = parse_gpt4_stepwise_response(response)

            # Append assistant response to history for next step
            history = messages + [{'role': 'assistant', 'content': pred}]

            action_golds.append(sgold)
            action_preds.append(1 if pred == 'A' else 0)
            if sgold != (1 if pred == 'A' else 0): step_mistake = True

            if pred == 'B':
                plan_pred = 0; stopped = True; break

        if not stopped:
            plan_pred = goal_check_from_records(recs, domain, goal_facts)

        plan_golds.append(plan_gold)
        plan_preds_out.append(plan_pred)

        if plan_gold != plan_pred:
            if step_mistake: err_action += 1
            else:            err_goal   += 1

        # Save checkpoint after each plan
        done_pids_sw[pid] = {"plan_id": pid, "plan_pred": plan_pred,
                              "plan_gold": plan_gold,
                              "action_preds": action_preds[-len(recs):],
                              "action_golds": action_golds[-len(recs):]}
        with open(ckpt_path, "w") as _f:
            json.dump(list(done_pids_sw.values()), _f)
        if (plan_idx + 1) % 10 == 0:
            print(f'    progress: {plan_idx+1}/{len(plan_ids)} plans')

    acc_plan = accuracy_score(plan_golds, plan_preds_out)
    f1_plan  = f1_score(plan_golds, plan_preds_out, average='macro', zero_division=0)
    cm_plan  = confusion_matrix(plan_golds, plan_preds_out, labels=[0, 1])
    acc_act  = accuracy_score(action_golds, action_preds) if action_golds else None
    f1_act   = f1_score(action_golds, action_preds, average='macro', zero_division=0) if action_golds else None
    cm_act   = confusion_matrix(action_golds, action_preds, labels=[0, 1]) if action_golds else None
    n_err    = sum(g != p for g, p in zip(plan_golds, plan_preds_out))

    print(f'    plan_acc={acc_plan:.4f} plan_f1={f1_plan:.4f}')

    return {
        'cond': 'G5_stepwise', 'domain': domain, 'eval_type': 'stepwise',
        'n_plans': len(plan_golds), 'n_actions': len(action_golds),
        'plan_acc': round(acc_plan, 4), 'plan_f1': round(f1_plan, 4),
        'plan_cm': cm_plan.tolist(),
        'action_acc': round(acc_act, 4) if acc_act is not None else None,
        'action_f1':  round(f1_act,  4) if f1_act  is not None else None,
        'action_cm':  cm_act.tolist()    if cm_act  is not None else None,
        'n_errors': n_err, 'err_action': err_action, 'err_goal': err_goal,
        'plan_golds': plan_golds, 'plan_preds': plan_preds_out,
        'action_golds': action_golds, 'action_preds': action_preds,
    }

print('G_stepwise evaluator: OK')

In [ ]:
# ── DEBUG SMOKE TEST — run this before full evaluation ───────────────────────
# Verifies: data loading, field presence, few-shot selection, GPT-4 API (1 call),
# adapter paths, prompt building, parse functions. Takes ~30 sec.

def run_smoke_test():
    import traceback

    # 1. Data files exist
    assert os.path.exists(SW_TEST_PATH), f'MISSING: {SW_TEST_PATH}'
    assert os.path.exists(DI_TEST_PATH), f'MISSING: {DI_TEST_PATH}'

    # 2. All domains have records
    for d in DOMAINS:
        assert len(sw_by_domain[d]) > 0, f'sw_by_domain[{d}] is empty'
        assert len(di_by_domain[d]) > 0, f'di_by_domain[{d}] is empty'

    # 3. Required fields in stepwise records
    sw_required = ['plan_id','step_idx','label','label_int','prompt','action',
                   'state_facts','rules','conclusion','goal_facts','plan_label']
    for d in DOMAINS:
        rec = sw_by_domain[d][0]
        for f in sw_required:
            assert f in rec, f'sw record[{d}] missing field: {f}'
        assert rec['label'] in ('A','B'),    f'sw label not A/B in {d}: {rec["label"]}'
        assert rec['label_int'] in (0,1),    f'sw label_int not 0/1 in {d}'
        assert isinstance(rec['rules'], list),       f'rules not list in {d}'
        assert isinstance(rec['state_facts'], list), f'state_facts not list in {d}'
        assert isinstance(rec['goal_facts'], list),  f'goal_facts not list in {d}'

    # 4. Required fields in direct records
    di_required = ['plan_id','label_int','prompt','rules','init_facts','goal_facts','actions']
    for d in DOMAINS:
        rec = di_by_domain[d][0]
        for f in di_required:
            assert f in rec, f'di record[{d}] missing field: {f}'
        assert rec['label_int'] in (0,1),          f'di label_int not 0/1 in {d}'
        assert isinstance(rec['actions'], list),   f'actions not list in {d}'
        assert isinstance(rec['init_facts'], list),f'init_facts not list in {d}'

    # 5. Few-shot example selection
    for d in DOMAINS:
        ex = gpt4_direct_examples[d]
        assert ex['valid']['label_int']   == 1, f'few-shot valid is not valid in {d}'
        assert ex['invalid']['label_int'] == 0, f'few-shot invalid is not invalid in {d}'
        assert ex['valid']['plan_id']   not in gpt4_direct_exclude[d] or True  # it IS in exclude
        assert len(gpt4_direct_exclude[d]) == 2, f'expected 2 excluded plan_ids in {d}'
        # Stepwise examples: valid should have at least 1 step
        assert len(gpt4_stepwise_examples[d]['valid'])   > 0, f'no valid sw steps in {d}'
        # Invalid should have at least 1 step with label B
        b_steps = [s for s in gpt4_stepwise_examples[d]['invalid'] if s['label']=='B']
        if len(b_steps) == 0:
            sw_inv = gpt4_stepwise_examples[d]['invalid']
            print(f'  [DEBUG] {d} invalid sw steps: {len(sw_inv)} total, '
                  f'labels={[s["label"] for s in sw_inv]}, '
                  f'plan_ids={list({s["plan_id"] for s in sw_inv})}')
        assert len(b_steps) > 0, f'no B-label step in invalid sw example for {d}'

    # 6. Prompt building
    d = DOMAINS[0]
    sw_rec = sw_by_domain[d][0]
    di_rec = di_by_domain[d][0]
    p_sw = build_stepwise_prompt(sw_rec['rules'], sw_rec['state_facts'], sw_rec['conclusion'])
    assert 'Task:' in p_sw and 'Premises:' in p_sw and 'Conclusion:' in p_sw, 'stepwise prompt malformed'
    p_di = build_direct_prompt_gpt4(di_rec['rules'], di_rec['init_facts'], di_rec['goal_facts'], di_rec['actions'])
    assert 'Task:' in p_di and 'Rules:' in p_di and 'Plan:' in p_di, 'direct prompt malformed'
    assert 'Step:' in p_di, 'direct GPT-4 prompt missing Step: in output format'

    # 7. GPT-4 message building
    msgs_direct = build_gpt4_direct_messages(d, di_rec)
    assert msgs_direct[0]['role'] == 'system',    'first message must be system'
    assert msgs_direct[-1]['role'] == 'user',     'last message must be user (test query)'
    assert len(msgs_direct) == 6,                 f'expected 6 messages (system+2 few-shot pairs+test), got {len(msgs_direct)}'
    # Check few-shot assistant answers have correct format
    fs_answers = [m['content'] for m in msgs_direct if m['role']=='assistant']
    for ans in fs_answers:
        assert re.match(r'Answer: [AB], Step: \d+', ans), f'bad few-shot answer format: {ans}'

    msgs_sw_first = build_gpt4_stepwise_messages(d, sw_rec['rules'], sw_rec['state_facts'],
                                                  sw_rec['conclusion'], conversation_history=None)
    assert msgs_sw_first[0]['role'] == 'system',  'first stepwise message must be system'
    assert msgs_sw_first[-1]['role'] == 'user',   'last stepwise message must be user'

    # 8. Parse functions
    label, step = parse_gpt4_direct_response('Answer: B, Step: 3')
    assert label == 'B' and step == 3,    f'parse failed: got {label},{step}'
    label, step = parse_gpt4_direct_response('Answer: A, Step: 0')
    assert label == 'A' and step == 0,    f'parse failed: got {label},{step}'
    label, step = parse_gpt4_direct_response('')
    assert label == 'EMPTY' and step is None, f'parse empty failed: got {label},{step}'
    assert normalize_pred('A blah') == 'A'
    assert normalize_pred('B blah') == 'B'
    assert normalize_pred('')       == 'B'
    assert normalize_pred('xyz')    == 'B'

    # 9. GPT-4 API test — direct call bypassing retry logic
    # Uses a short timeout so smoke test fails fast instead of waiting minutes
    print(f'  Testing {GPT5_MODEL} API (1 direct call)...')
    try:
        _client = openai.OpenAI(api_key=openai.api_key)
        _resp = _client.chat.completions.create(
            model=GPT5_MODEL,
            messages=[{'role':'user','content':'Reply with exactly: A'}],
            max_tokens=5,
            temperature=0,
        )
        _text = _resp.choices[0].message.content.strip()
        print(f'  API OK. Model={GPT5_MODEL} Response={repr(_text)}')
    except openai.RateLimitError:
        print(f'  [WARN] Rate limited on smoke test — but API key is valid.')
        print(f'  Evaluation will use exponential backoff (base_delay=60s).')
        print(f'  This is expected on lower tiers. Proceeding.')
    except openai.AuthenticationError as e:
        raise AssertionError(f'API key invalid: {e}') from e
    except Exception as e:
        print(f'  [WARN] Unexpected error in API test: {e}')
        print(f'  If this is a rate limit, evaluation will still work with backoff.')

    # 10. Adapter paths
    # No Flan-T5 adapters needed — GPT-5.4 API only

    print('\nSmoke test PASSED')

run_smoke_test()


## 10. Run All Conditions

In [ ]:
# ── Delete stale checkpoints from failed runs ────────────────────────────
# Run this if a previous run had API errors (e.g. wrong max_tokens param)
# to prevent bad cached results from being replayed.
import glob
stale = glob.glob(os.path.join(RESULTS_DIR, 'ckpt_G5_*.json'))
if stale:
    for f in stale:
        os.remove(f)
        print(f'Deleted stale checkpoint: {f}')
    print(f'Deleted {len(stale)} stale checkpoints.')
else:
    print('No stale checkpoints found.')

In [ ]:
all_results = {}

# ── G_direct (GPT-4, direct few-shot) ─────────────────────────────────────────
print(f'\n{"="*60}\nCONDITION: G_direct\n{"="*60}')
all_results['G5_direct'] = {}
for domain in DOMAINS:
    records = di_by_domain[domain]
    if not records: continue
    r = evaluate_g_direct(records, domain)
    all_results['G5_direct'][domain] = r
print('G_direct complete')

In [ ]:
# ── G_stepwise (GPT-4, stepwise few-shot) ────────────────────────────────────
print(f'\n{"="*60}\nCONDITION: G_stepwise\n{"="*60}')
all_results['G5_stepwise'] = {}
for domain in DOMAINS:
    records = sw_by_domain[domain]
    if not records: continue
    r = evaluate_g_stepwise(records, domain)
    all_results['G5_stepwise'][domain] = r
print('G_stepwise complete')

## 11. Reports

In [ ]:
def print_domain_report(r):
    cond  = r['cond']; domain = r['domain']; etype = r['eval_type']
    print(f'\n{"="*65}')
    print(f'{cond} | {domain.upper()} | {etype.upper()}')
    print(f'{"="*65}')
    print(f'\nPLAN-LEVEL  n={r["n_plans"]}  acc={r["plan_acc"]:.4f}  f1={r["plan_f1"]:.4f}')
    print(classification_report(r['plan_golds'], r['plan_preds'],
          target_names=['Invalid', 'Valid'], labels=[0, 1], zero_division=0))
    cm = r['plan_cm']
    print(f'  CM                Pred:Invalid  Pred:Valid')
    print(f'  Gold:Invalid      {cm[0][0]:12d}  {cm[0][1]:9d}')
    print(f'  Gold:Valid        {cm[1][0]:12d}  {cm[1][1]:9d}')
    if r.get('action_acc') is not None:
        print(f'\nACTION-LEVEL  n={r.get("n_actions", len(r["action_golds"]))}')
        print(f'  acc={r["action_acc"]:.4f}  f1={r["action_f1"]:.4f}')
        acm = r['action_cm']
        if acm:
            print(f'  CM                   Pred:NotExec  Pred:Exec')
            print(f'  Gold:Not Executable  {acm[0][0]:12d}  {acm[0][1]:9d}')
            print(f'  Gold:Executable      {acm[1][0]:12d}  {acm[1][1]:9d}')
    if r.get('failing_step_acc') is not None:
        print(f'\nFAILING-STEP ACCURACY: {r["failing_step_acc"]:.4f} '
              f'(n={r["n_failing_step_evaluated"]} invalid plans with known fail step)')
    if etype == 'stepwise' and r.get('n_errors') is not None:
        ne = r['n_errors']
        print(f'\nERROR ANALYSIS: {ne} misclassified plans')
        print(f'  action_mistake    : {r["err_action"]} ({r["err_action"]/max(1,ne)*100:.1f}%)')
        print(f'  goal_check_mistake: {r["err_goal"]}  ({r["err_goal"]/max(1,ne)*100:.1f}%)')

def merged_result(results_list, group_name, cond):
    pg = [g for r in results_list for g in r['plan_golds']]
    pp = [p for r in results_list for p in r['plan_preds']]
    acc = accuracy_score(pg, pp)
    f1  = f1_score(pg, pp, average='macro', zero_division=0)
    cm  = confusion_matrix(pg, pp, labels=[0, 1])
    merged = {
        'cond': cond, 'domain': group_name,
        'eval_type': results_list[0]['eval_type'],
        'n_plans': len(pg),
        'plan_acc': round(acc, 4), 'plan_f1': round(f1, 4),
        'plan_cm': cm.tolist(), 'plan_golds': pg, 'plan_preds': pp,
    }
    ag = [g for r in results_list if r.get('action_golds') for g in r['action_golds']]
    ap = [p for r in results_list if r.get('action_preds') for p in r['action_preds']]
    if ag:
        a_acc = accuracy_score(ag, ap)
        a_f1  = f1_score(ag, ap, average='macro', zero_division=0)
        a_cm  = confusion_matrix(ag, ap, labels=[0, 1])
        merged.update({
            'action_golds': ag, 'action_preds': ap,
            'action_acc': round(a_acc, 4), 'action_f1': round(a_f1, 4),
            'action_cm': a_cm.tolist(), 'n_actions': len(ag),
            'n_errors':    sum(r.get('n_errors',    0) for r in results_list),
            'err_action':  sum(r.get('err_action',  0) for r in results_list),
            'err_goal':    sum(r.get('err_goal',     0) for r in results_list),
        })
    # Failing step (G_direct only)
    fsg = [g for r in results_list if r.get('failing_step_golds') for g in r['failing_step_golds']]
    fsp = [p for r in results_list if r.get('failing_step_preds') for p in r['failing_step_preds']]
    if fsg:
        fs_acc = sum(g == p for g, p in zip(fsg, fsp)) / max(1, len(fsg))
        merged['failing_step_acc'] = round(fs_acc, 4)
        merged['n_failing_step_evaluated'] = len(fsg)
        merged['failing_step_golds'] = fsg
        merged['failing_step_preds'] = fsp
    return merged

ALL_CONDS = [c for c in ['G5_direct', 'G5_stepwise']
             if c in all_results]

for cond in ALL_CONDS:
    for domain in DOMAINS:
        if domain in all_results.get(cond, {}):
            print_domain_report(all_results[cond][domain])

print('\n' + '─'*65 + '\nMERGED GROUP RESULTS')
for cond in ALL_CONDS:
    if not all_results.get(cond): continue
    for group, subs in MERGE_GROUPS.items():
        sub_results = [all_results[cond][d] for d in subs if d in all_results.get(cond, {})]
        if sub_results:
            mr   = merged_result(sub_results, group, cond)
            line = f'  {cond:15s} | {group:12s} | plan_acc={mr["plan_acc"]:.4f} plan_f1={mr["plan_f1"]:.4f}'
            if mr.get('action_acc'):
                line += f' | act_acc={mr["action_acc"]:.4f} act_f1={mr["action_f1"]:.4f}'
            if mr.get('failing_step_acc') is not None:
                line += f' | fail_step_acc={mr["failing_step_acc"]:.4f}'
            print(line)

## 12. Summary Tables

In [ ]:
SEP = '─' * 80

def summary_table(metric, eval_type, label):
    conds = [c for c in ALL_CONDS
             if any(all_results.get(c, {}).get(d, {}).get('eval_type') == eval_type
                    for d in DOMAINS)]
    if not conds: return
    print(f'\n{SEP}\n{label}')
    print(f'{"Cond":15s}  ' + '  '.join(f'{d[:10]:>10s}' for d in DOMAINS) + f'  {"Mean":>8s}')
    print(SEP)
    for cond in conds:
        vals  = [all_results.get(cond, {}).get(d, {}).get(metric) for d in DOMAINS]
        valid = [v for v in vals if v is not None]
        mean  = sum(valid) / len(valid) if valid else float('nan')
        row   = f'{cond:15s}  ' + '  '.join(
            f'{v:>10.4f}' if v is not None else f'{"N/A":>10s}' for v in vals)
        print(row + f'  {mean:>8.4f}')

summary_table('plan_f1',          'direct',   'DIRECT — Plan F1-macro (G_direct)')
summary_table('plan_acc',         'direct',   'DIRECT — Plan Accuracy (G_direct)')
summary_table('action_f1',        'direct',   'DIRECT — Action F1-macro (G_direct)')
summary_table('failing_step_acc', 'direct',   'DIRECT — Failing Step Accuracy (G_direct)')
summary_table('plan_f1',          'stepwise', 'STEPWISE — Plan F1-macro')
summary_table('plan_acc',         'stepwise', 'STEPWISE — Plan Accuracy')
summary_table('action_f1',        'stepwise', 'STEPWISE — Action F1-macro')
summary_table('action_acc',       'stepwise', 'STEPWISE — Action Accuracy')

## 13. Save Results

In [ ]:
SKIP_KEYS = {'plan_golds', 'plan_preds', 'action_golds', 'action_preds',
             'failing_step_golds', 'failing_step_preds'}

def serialise(r):
    return {k: v for k, v in r.items() if k not in SKIP_KEYS}

save_dict = {
    cond: {domain: serialise(r) for domain, r in domain_results.items()}
    for cond, domain_results in all_results.items()
}

# Add merged groups
for cond in ALL_CONDS:
    if not all_results.get(cond): continue
    save_dict[cond]['_merged'] = {}
    for group, subs in MERGE_GROUPS.items():
        sub_results = [all_results[cond][d] for d in subs if d in all_results.get(cond, {})]
        if sub_results:
            mr = merged_result(sub_results, group, cond)
            save_dict[cond]['_merged'][group] = serialise(mr)

results_path = os.path.join(RESULTS_DIR, 'phase6_results.json')
with open(results_path, 'w') as f:
    json.dump(save_dict, f, indent=2, default=str)
print(f'Results saved → {results_path}')

In [ ]:
# Save raw prediction arrays for additional metrics
RAW_KEYS = {'plan_golds','plan_preds','action_golds','action_preds',
            'failing_step_golds','failing_step_preds'}
raw_dict = {cond: {d: {k:v for k,v in r.items() if k in RAW_KEYS}
                   for d,r in dom_res.items()}
            for cond, dom_res in all_results.items()}
raw_path = os.path.join(RESULTS_DIR, 'raw_preds_gpt5.json')
with open(raw_path, 'w') as f:
    json.dump(raw_dict, f)
print(f'Raw predictions saved -> {raw_path}')